# T09: Streaming Events

## What You'll Learn

Real-time data pipelines need realistic event streams to test. In this notebook you will:

1. **Configure** a stream with events per second and max event limits
2. **Stream** retail order events to a JSONL file
3. **Inspect** the output events
4. **Add** anomaly injection via burst windows
5. **Demonstrate** out-of-order event delivery

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle`

## Time Estimate

**~5 minutes** from start to finish.

In [1]:
# Uncomment the line below if you're running in an environment
# where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Configure the Stream

### What we're about to do
We'll create a `StreamConfig` that controls the throughput and behavior of our event stream:
- **events_per_second**: target throughput (100 events/sec)
- **max_events**: hard stop after 500 events
- **out_of_order_fraction**: 5% of events will arrive with timestamps earlier than preceding events

### Why this matters
Real event streams are never perfectly ordered. Network delays, retries, and partition rebalancing all cause events to arrive out of sequence. Testing with out-of-order data ensures your pipeline handles late arrivals correctly.

### What to expect
A configured stream ready to emit events.

In [2]:
from sqllocks_spindle import SpindleStreamer, StreamConfig, FileSink, RetailDomain

config = StreamConfig(
    events_per_second=100,
    max_events=500,
    out_of_order_fraction=0.05,
)

print(f"Events per second: {config.events_per_second}")
print(f"Max events: {config.max_events}")
print(f"Out-of-order fraction: {config.out_of_order_fraction}")

Events per second: 100
Max events: 500
Out-of-order fraction: 0.05


## Stream Order Events to a File

### What we're about to do
We'll create a `FileSink` that writes events as newline-delimited JSON (JSONL), then start the streamer targeting the `order` event type from the Retail domain.

### Why this matters
JSONL is the standard format for streaming ingestion into systems like Azure Event Hubs, Kafka, and Fabric Eventstreams. Writing to a file first lets you inspect the data before connecting to a live sink.

### What to expect
500 events written to `./stream_demo/events.jsonl`, with a summary showing total events and out-of-order count.

In [3]:
import os
os.makedirs("./stream_demo", exist_ok=True)

sink = FileSink("./stream_demo/events.jsonl", mode="w")
streamer = SpindleStreamer(
    domain=RetailDomain(),
    sink=sink,
    config=config,
    scale="small",
    seed=42,
)

result = streamer.stream("order")
sink.close()

print(f"Events sent: {result.events_sent}")
print(f"Out-of-order events: {result.out_of_order_count}")
print(f"Duration: {result.elapsed_seconds:.2f}s")

Events sent: 500
Out-of-order events: 250
Duration: 0.00s


## Inspect the Output Events

### What we're about to do
We'll read the first 5 events from the JSONL file and pretty-print them. This lets you see the event schema, timestamps, and payload structure.

### Why this matters
Before wiring a stream into a production pipeline, you need to understand the event shape. Field names, data types, and nesting all affect how you parse and transform downstream.

### What to expect
Five JSON objects, each representing a retail order event with timestamps, customer references, and order details.

In [4]:
import json

print("=== First 5 Events ===")
with open("./stream_demo/events.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        event = json.loads(line)
        print(json.dumps(event, indent=2, default=str))
        print()

=== First 5 Events ===
{
  "order_id": 2539,
  "customer_id": 452,
  "store_id": 140,
  "shipping_address_id": 1202,
  "promotion_id": 62,
  "order_date": "2022-02-22 05:34:49.576566715",
  "status": "completed",
  "order_total": 122.33,
  "_spindle_table": "order",
  "_spindle_seq": 2538,
  "_spindle_event_time": "2022-02-22 05:34:49.576566715"
}

{
  "order_id": 777,
  "customer_id": 280,
  "store_id": 7,
  "shipping_address_id": null,
  "promotion_id": 99,
  "order_date": "2022-02-25 13:52:33",
  "status": "shipped",
  "order_total": 96.31,
  "_spindle_table": "order",
  "_spindle_seq": 776,
  "_spindle_event_time": "2022-02-25 13:52:33"
}

{
  "order_id": 4454,
  "customer_id": 441,
  "store_id": 41,
  "shipping_address_id": 1229,
  "promotion_id": null,
  "order_date": "2022-03-02 00:44:05.553639072",
  "status": "completed",
  "order_total": 133.65,
  "_spindle_table": "order",
  "_spindle_seq": 4453,
  "_spindle_event_time": "2022-03-02 00:44:05.553639072"
}

{
  "order_id": 229

## Detect Out-of-Order Events

### What we're about to do
We'll scan the event stream for timestamps that are earlier than the preceding event. These are the out-of-order (OOO) events we configured with `out_of_order_fraction=0.05`.

### Why this matters
Late-arriving events are a major source of bugs in streaming pipelines. Windowed aggregations, watermarks, and exactly-once processing all depend on handling OOO events correctly. By injecting them synthetically, you can test your pipeline's resilience.

### What to expect
A count and examples of events whose timestamps are out of sequence.

In [5]:
import json

events = []
with open("./stream_demo/events.jsonl") as f:
    for line in f:
        events.append(json.loads(line))

ooo_count = 0
ooo_examples = []
for i in range(1, len(events)):
    current_ts = events[i].get("event_time") or events[i].get("timestamp", "")
    previous_ts = events[i - 1].get("event_time") or events[i - 1].get("timestamp", "")
    if current_ts < previous_ts:
        ooo_count += 1
        if len(ooo_examples) < 3:
            ooo_examples.append((i, previous_ts, current_ts))

print(f"Total events: {len(events)}")
print(f"Out-of-order events detected: {ooo_count}")
print(f"OOO rate: {ooo_count / len(events) * 100:.1f}%")

if ooo_examples:
    print("\n=== Example OOO Events ===")
    for idx, prev_ts, curr_ts in ooo_examples:
        print(f"  Event {idx}: {curr_ts} arrived after {prev_ts}")

Total events: 500
Out-of-order events detected: 0
OOO rate: 0.0%


## Stream to Console with Burst Window

### What we're about to do
We'll create a second stream using a `ConsoleSink` and add a `BurstWindow` — a period where the event rate spikes dramatically, simulating flash sales or traffic surges.

### Why this matters
Production systems experience traffic bursts that can overwhelm downstream consumers. Testing with burst windows ensures your pipeline can handle sudden throughput spikes without dropping events or falling behind.

### What to expect
A short burst of events printed to the console, with a summary showing elevated throughput during the burst window.

In [6]:
from sqllocks_spindle import ConsoleSink, BurstWindow

burst_config = StreamConfig(
    events_per_second=50,
    max_events=100,
    out_of_order_fraction=0.0,
    burst_windows=[BurstWindow(start_offset_seconds=30, duration_seconds=30, multiplier=5.0)],
)

console_sink = ConsoleSink(indent=2)
burst_streamer = SpindleStreamer(
    domain=RetailDomain(),
    sink=console_sink,
    config=burst_config,
    scale="small",
    seed=99,
)

burst_result = burst_streamer.stream("order")

print(f"\n=== Burst Stream Summary ===")
print(f"Events sent: {burst_result.events_sent}")
print(f"Duration: {burst_result.elapsed_seconds:.2f}s")
print(f"Effective rate: {burst_result.events_per_second_actual:.0f} events/sec")

{
  "order_id": 1167,
  "customer_id": 11,
  "store_id": 1,
  "shipping_address_id": 271,
  "promotion_id": null,
  "order_date": "2022-01-30 09:28:51.037740866",
  "status": "completed",
  "order_total": 152.85,
  "_spindle_table": "order",
  "_spindle_seq": 1166,
  "_spindle_event_time": "2022-01-30 09:28:51.037740866"
}
{
  "order_id": 4377,
  "customer_id": 11,
  "store_id": 5,
  "shipping_address_id": 1320,
  "promotion_id": null,
  "order_date": "2022-01-30 09:28:51.037740866",
  "status": "completed",
  "order_total": 28.1,
  "_spindle_table": "order",
  "_spindle_seq": 4376,
  "_spindle_event_time": "2022-01-30 09:28:51.037740866"
}
{
  "order_id": 4519,
  "customer_id": 660,
  "store_id": 3,
  "shipping_address_id": 443,
  "promotion_id": 151,
  "order_date": "2022-02-06 23:41:49",
  "status": "shipped",
  "order_total": 151.74,
  "_spindle_table": "order",
  "_spindle_seq": 4518,
  "_spindle_event_time": "2022-02-06 23:41:49"
}
{
  "order_id": 1520,
  "customer_id": 20,
  "st

## What's Next?

You've streamed events with configurable throughput, out-of-order delivery, and burst windows. Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T10: File-Drop Simulation** | Simulate daily file drops with manifests, done flags, and late arrivals |
| **T11: Chaos Engineering** | Inject schema drift, null corruption, and referential breakage |
| **T12: Validation Gates** | Catch data quality issues with automated gate checks |
| **[F04 — Real-Time Streaming](../fabric-scenarios/F04_realtime_streaming.ipynb)** | Stream capital markets events with burst windows, anomaly injection, and Eventstream integration |

Happy streaming!